In [1]:
import chromadb
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='paraphrase-multilingual-MiniLM-L12-v2'
)
client = chromadb.PersistentClient(path='../data/chroma_index')
coleccion = client.get_collection(name='corpus_laliga', embedding_function=embedding_fn)
print("Chunks disponibles:", coleccion.count())

/home/angel/sistema-multiagente-futbol/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 550.98it/s]


Chunks disponibles: 997


In [2]:
def recuperar_contexto(pregunta: str, k: int = 4) -> list[dict]:
    res = coleccion.query(query_texts=[pregunta], n_results=k)
    return [
        {'texto': t, 'documento': m['documento'], 'seccion': m['seccion'], 'distancia': d}
        for t, m, d in zip(res['documents'][0], res['metadatas'][0], res['distances'][0])
    ]

In [3]:
def construir_prompt(pregunta: str, fragmentos: list[dict]) -> str:
    contexto = "\n\n".join(
        f"[Fuente {i+1}: {f['documento']} | {f['seccion'] or 'sin sección'}]\n{f['texto']}"
        for i, f in enumerate(fragmentos)
    )
    return f"""Eres el agente de conocimiento de un sistema de analítica de LaLiga. Responde la pregunta del usuario usando ÚNICAMENTE la información de los fragmentos de abajo.

Reglas:
- Si los fragmentos no contienen la respuesta, di claramente que no encontraste esa información en los documentos disponibles. No inventes.
- Cita las fuentes que uses con el formato [Fuente N].
- Responde en español, de forma clara y concisa.

FRAGMENTOS:
{contexto}

PREGUNTA: {pregunta}"""

In [4]:
pregunta = "¿Cuántos suplentes se pueden usar en un partido?"
fragmentos = recuperar_contexto(pregunta)
print(construir_prompt(pregunta, fragmentos))

Eres el agente de conocimiento de un sistema de analítica de LaLiga. Responde la pregunta del usuario usando ÚNICAMENTE la información de los fragmentos de abajo.

Reglas:
- Si los fragmentos no contienen la respuesta, di claramente que no encontraste esa información en los documentos disponibles. No inventes.
- Cita las fuentes que uses con el formato [Fuente N].
- Responde en español, de forma clara y concisa.

FRAGMENTOS:
[Fuente 1: reglas_juego_cambios | Regla 14. El penal (p. 149, 150)]
221
Terreno de juego (→ field of play [pitch]) 
Superficie delimitada por las líneas de banda y las de meta y, de utilizarse, las 
redes de las porterías.
Tiempo adicional (→ additional time) 
Minutos añadidos al final de cada uno de los periodos para compensar el tiempo 
perdido, p. ej. por sustituciones, lesiones, sanciones disciplinarias, celebración 
de goles, etc.
Tiempo suplementario (→ extra time) 
Procedimiento empleado a la conclusión del tiempo reglamentario de un 
partido acabado en empa

In [5]:
for f in recuperar_contexto("¿Cuál es la receta de la paella valenciana?"):
    print(f"[{f['documento']} | {f['seccion']}] distancia: {f['distancia']:.3f}")

[primera_division | ] distancia: 0.538
[clasico | ] distancia: 0.553
[clasico | ] distancia: 0.563
[clasico | ] distancia: 0.571


In [6]:
import time
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:7b", temperature=0)

def responder(pregunta: str, k: int = 4) -> dict:
    fragmentos = recuperar_contexto(pregunta, k)
    respuesta = llm.invoke(construir_prompt(pregunta, fragmentos))
    return {'respuesta': respuesta.content, 'fuentes': fragmentos}

llm.invoke("Hola")   # calentamiento: la primera llamada carga el modelo en memoria y tarda más

t0 = time.time()
r = responder("¿Cuántos suplentes se pueden usar en un partido?")
print(r['respuesta'])
print(f"\nTiempo: {time.time() - t0:.1f} s")

Según la información proporcionada en los fragmentos, en un partido de selecciones nacionales absolutas, se pueden utilizar un máximo de seis ocho suplentes, a menos que los dos equipos acuerden utilizar más hasta un máximo de once. En partidos amistosos internacionales entre selecciones absolutas, se permite usar hasta ocho suplentes inscritos como tal, a menos que los dos equipos acuerden utilizar más, hasta un máximo de once.

[Fuente 2: reglas_juego_cambios | Regla 3. Los jugadores]

Tiempo: 9.5 s


In [7]:
print(responder("¿Qué es el fuera de juego?")['respuesta'])

El fuera de juego ocurre cuando un jugador se encuentra en una posición que lo hace susceptible a sanciones, según las reglas del fútbol. Un jugador está en posición de fuera de juego si:

1. Su cabeza, tronco o pierna (incluyendo el pie) se encuentran, total o parcialmente, en la mitad del terreno de juego del adversario (excluyendo la línea central).
2. Se encuentran más cerca de la línea de meta contraria que el balón y el penúltimo adversario.

Sin embargo, no estará en fuera de juego si se encuentra a la misma altura que el penúltimo adversario o los dos últimos adversarios.

Solo se sancionará a un jugador en posición de fuera de juego en el momento en que el balón toca o es jugado por un compañero de equipo si llega a participar en el juego de forma activa, interfiriendo en el juego o ganando ventaja de dicha posición.

[Fuente 1: reglas_juego | Regla 3. Los jugadores]
[Fuente 2: reglamento | Artículo 34. Fuera de juego]


In [8]:
print(responder("¿Cuál es la receta de la paella valenciana?")['respuesta'])

No encontré información sobre la receta de la paella valenciana en los fragmentos proporcionados. Los fragmentos están relacionados con LaLiga y el fútbol, no con la cocina o recetas específicas. [Fuente 1 a 4: primera_division, clasico, clasico, clasico]


In [9]:
from collections import Counter

# 1. ¿Qué tan repartidas están las etiquetas de sección en reglas_juego?
m = coleccion.get(where={"documento": "reglas_juego"}, include=["metadatas"])['metadatas']
conteo = Counter(x['seccion'] for x in m)
print(len(conteo), "secciones distintas en", len(m), "chunks")
for sec, n in conteo.most_common(15):
    print(n, '|', sec)

# 2. ¿Cuántos chunks del reglamento mencionan GENUINE?
g = coleccion.get(where={"documento": "reglamento"}, include=["documents"])['documents']
print("\n", sum('GENUINE' in d.upper() for d in g), "de", len(g), "chunks del reglamento mencionan GENUINE")

10 secciones distintas en 282 chunks
259 | Regla 3. Los jugadores
15 | Regla 5. El árbitro
1 | Regla 4. El equipamiento de los jugadores
1 | Regla 6. Los otros miembros del equipo arbitral
1 | Regla 8. Inicio y reanudación del juego
1 | Regla 10. El resultado de un partido
1 | Regla 12. Faltas y conducta incorrecta
1 | Regla 14. El penal
1 | Regla 15. El saque de banda
1 | Regla 16. El saque de meta

 48 de 121 chunks del reglamento mencionan GENUINE


In [10]:
import re, time, pymupdf
from collections import Counter

# A) ¿Con qué formato aparecen los títulos de regla dentro de los chunks?
docs = coleccion.get(where={"documento": "reglas_juego"}, include=["documents"])['documents']
titulos = Counter()
for d in docs:
    for linea in d.split('\n'):
        if re.match(r'\s*Regla\s*\d+', linea):
            titulos[linea.strip()[:50]] += 1
print("A) líneas que empiezan con 'Regla N':")
for t, n in titulos.most_common(25):
    print(f"   {n} | {t!r}")

# B) ¿El pie de página original dice en qué capítulo estás?
pdf = pymupdf.open('../data/Reglas de Juego 2026_27_paginas individuales.pdf')
print("\nB) final de algunas páginas (antes de limpiar):")
for p in (60, 100, 150, 200):
    print(f"   p.{p+1}:", repr(pdf[p].get_text()[-120:]))
pdf.close()

# C) ¿Qué diría el sistema sobre la duración de un partido? (y cuánto tarda)
t0 = time.time()
print("\nC)", responder("¿Cuánto dura un partido de LaLiga?")['respuesta'])
print(f"   Tiempo: {time.time() - t0:.1f} s")

A) líneas que empiezan con 'Regla N':
   3 | 'Regla 3. Los jugadores'
   2 | 'Regla 5. El árbitro'
   1 | 'Regla1'
   1 | 'Regla7'
   1 | 'Regla8'
   1 | 'Regla9'
   1 | 'Regla 4. El equipamiento de los jugadores'
   1 | 'Regla 6. Los otros miembros del equipo arbitral'
   1 | 'Regla 8. Inicio y reanudación del juego'
   1 | 'Regla 10. El resultado de un partido'
   1 | 'Regla 12. Faltas y conducta incorrecta'
   1 | 'Regla 14. El penal'
   1 | 'Regla 15. El saque de banda'
   1 | 'Regla 16. El saque de meta'

B) final de algunas páginas (antes de limpiar):
   p.61: '61\n'
   p.101: 'tivo al reanudarse \nel juego (p. ej. debido a una interferencia causada por un agente externo).\nLa duración del partido\n'
   p.151: '151\n'
   p.201: 'tación)\n•\u2002\x07Evitar una ocasión manifiesta de gol (especialmente el lugar de la \nfalta y la posición de otros jugadores).\n'

C) Un partido de LaLiga consta de cuatro tiempos de 10 minutos cada uno, con intermedios de 2 minutos y un descanso de 5 

In [11]:
q = "¿Cuánto dura un partido de LaLiga?"
for i in range(2):
    t0 = time.time()
    frag = recuperar_contexto(q, 4)
    t1 = time.time()
    resp = llm.invoke(construir_prompt(q, frag))
    t2 = time.time()
    md = resp.response_metadata
    print(f"Corrida {i+1}: búsqueda {t1-t0:.2f}s | LLM total {t2-t1:.1f}s | "
          f"carga {md.get('load_duration',0)/1e9:.1f}s | "
          f"leer prompt {md.get('prompt_eval_duration',0)/1e9:.1f}s ({md.get('prompt_eval_count')} tokens) | "
          f"generar {md.get('eval_duration',0)/1e9:.1f}s ({md.get('eval_count')} tokens)")

Corrida 1: búsqueda 0.17s | LLM total 11.7s | carga 0.0s | leer prompt 0.1s (1142 tokens) | generar 11.5s (147 tokens)
Corrida 2: búsqueda 0.05s | LLM total 9.9s | carga 0.0s | leer prompt 0.1s (1142 tokens) | generar 9.8s (147 tokens)


In [12]:
d = pymupdf.open('../data/Reglas de Juego 2026_27_paginas individuales.pdf')
toc = d.get_toc()
print("Marcadores en reglas de juego:", len(toc))
for nivel, titulo, pag in toc[:40]:
    print(nivel, '|', titulo, '| pág.', pag)
d.close()

r = pymupdf.open('../data/reglamento laliga 2025-2026.pdf')
print("\n--- portada del reglamento ---\n", r[0].get_text()[:500])
print("\n--- página 2 ---\n", r[1].get_text()[:600])
r.close()

Marcadores en reglas de juego: 0

--- portada del reglamento ---
  
  
 
REGLAMENTO 
TEMPORADA 2025/26 
 


--- página 2 ---
  
 
2 
ÍNDICE ESQUEMÁTICO 
Artículo 1. Finalidad y esencia de la competición ......................................................... 5 
Artículo 2. Objetivos .................................................................................................................. 5 
Artículo 3. Regulación aplicable ........................................................................................... 5 
Artículo 4. De los clubes participantes .............................................................................. 6 
Artículo 5. Fechas y sedes en las que se disputarán los encuentros .....


In [13]:
import subprocess
q = "¿Cuánto dura un partido de LaLiga?"
base = construir_prompt(q, recuperar_contexto(q, 4))
pruebas = {
    'A: actual': (dict(), base),
    'B: num_ctx 2048': (dict(num_ctx=2048), base),
    'C: num_ctx 2048 + respuesta corta': (dict(num_ctx=2048), base + "\n\nResponde en máximo 3 oraciones."),
}
for nombre, (kw, prompt) in pruebas.items():
    m = ChatOllama(model="qwen2.5:7b", temperature=0, **kw)
    m.invoke(prompt)                              # calentamiento: carga el modelo con esta configuración
    t0 = time.time(); r = m.invoke(prompt); t = time.time() - t0
    md = r.response_metadata
    n, dur = md.get('eval_count', 0), md.get('eval_duration', 1) / 1e9
    print(f"{nombre}: {t:.1f} s | {n} tokens a {n/dur:.1f} tok/s")
    print(subprocess.run(['ollama', 'ps'], capture_output=True, text=True).stdout)

A: actual: 14373.7 s | 147 tokens a 11.7 tok/s
NAME          ID              SIZE      PROCESSOR          CONTEXT    RUNNER      UNTIL              
qwen2.5:7b    845dbda0ea48    5.2 GB    18%/82% CPU/GPU    4096       llamacpp    4 minutes from now    

B: num_ctx 2048: 11.1 s | 146 tokens a 13.4 tok/s
NAME          ID              SIZE      PROCESSOR          CONTEXT    RUNNER      UNTIL              
qwen2.5:7b    845dbda0ea48    5.0 GB    16%/84% CPU/GPU    2048       llamacpp    4 minutes from now    

C: num_ctx 2048 + respuesta corta: 14.5 s | 128 tokens a 8.9 tok/s
NAME          ID              SIZE      PROCESSOR          CONTEXT    RUNNER      UNTIL              
qwen2.5:7b    845dbda0ea48    5.0 GB    16%/84% CPU/GPU    2048       llamacpp    4 minutes from now    



In [14]:
d = pymupdf.open('../data/Reglas de Juego 2026_27_paginas individuales.pdf')
for i, p in enumerate(d):
    t = p.get_text().strip()
    if len(t) < 250 and re.search(r'Regla\s*\d', t):
        print(i + 1, '|', repr(t[:100]))
d.close()

48 | 'Regla1'
58 | 'Regla2'
62 | 'Regla3'
72 | 'Regla4'
80 | 'Regla5'
89 | '89\nReglas de Juego 2026/27  |  Regla 5  |  El árbitro\nTarjeta roja o amarilla\nChequeo: un dedo a la '
92 | 'Regla6'
97 | '97\nSaque de esquina\nSaque de meta\nSaque de banda\npara el defensor\nSaque de banda\npara el atacante\nRe'
100 | 'Regla7'
104 | 'Regla8'
108 | 'Regla9'
110 | 'Regla10'
116 | 'Regla11'
122 | 'Regla12'
140 | 'Regla13'
144 | 'Regla14'
152 | 'Regla15'
156 | 'Regla16'
160 | 'Regla17'


In [15]:
r = pymupdf.open('../data/reglamento laliga 2025-2026.pdf')
print(r[4].get_text()[:1500])
r.close()


 
 
5 
 
CAPÍTULO I. DISPOSICIONES GENERALES 
Artículo 1. Finalidad y esencia de la competición 
LALIGA GENUINE Moeve es una iniciativa integradora de responsabilidad 
social que, mediante la organización de una competición formada por 
equipos del colectivo DI (personas con discapacidad intelectual), persigue 
contribuir al crecimiento personal e inclusión social de sus participantes a 
través de una experiencia de convivencia única complementada con la 
práctica del fútbol.  
Artículo 2. Objetivos   
Los objetivos perseguidos por LALIGA GENUINE Moeve son: 
a) Normalizar e impulsar la práctica del fútbol en el colectivo DI. 
b) Manifestar el compromiso del fútbol profesional con este proyecto 
integrador y socialmente responsable. 
c) Impulsar la creación de equipos GENUINE en la estructura deportiva 
de los clubes/SAD afiliados a LALIGA. 
Artículo 3. Regulación aplicable 
3.1. La regulación normativa, organización y desarrollo de la competición 
LALIGA GENUINE Moeve correspondiente a

In [16]:
d = pymupdf.open('../data/Reglas de Juego 2026_27_paginas individuales.pdf')
patron = re.compile(r'Reglas de Juego 2026/27\s*\|\s*([^\n]+)')

encabezados = {}
for i, p in enumerate(d):
    m = patron.search(p.get_text())
    if m:
        encabezados[i + 1] = re.sub(r'\s+', ' ', m.group(1)).strip()

print(len(encabezados), "de", len(d), "páginas traen encabezado con capítulo\n")
primeras = {}
for pag, h in encabezados.items():
    primeras.setdefault(h, pag)
for h, pag in list(primeras.items())[:45]:
    print(f"p.{pag} | {h}")
d.close()

110 de 276 páginas traen encabezado con capítulo

p.7 | Índice
p.11 | Acerca de las Reglas
p.17 | Observaciones y adaptaciones
p.49 | Regla 1 | El terreno de juego
p.59 | Regla 2 | El balón
p.63 | Regla 3 | Los jugadores
p.73 | Regla 4 | El equipamiento de los jugadores
p.81 | Regla 5 | El árbitro
p.93 | Regla 6 | Los otros miembros del equipo arbitral
p.101 | Regla 7 | La duración del partido
p.105 | Regla 8 | Inicio y reanudación del juego
p.109 | Regla 9 | Balón en juego
p.111 | Regla 10 | El resultado de un partido
p.117 | Regla 11 | El fuera de juego
p.123 | Regla 12 | Faltas y conducta incorrecta
p.141 | Regla 13 | Tiros libres
p.145 | Regla 14 | El penal
p.153 | Regla 15 | El saque de banda
p.157 | Regla 16 | El saque de meta
p.161 | Regla 17 | El saque de esquina
p.165 | Protocolo del VAR
p.177 | Programa de calidad de la FIFA
p.181 | Cambios en las Reglas 2026/27
p.211 | Glosario
p.225 | Directrices prácticas para los miembros del equipo arbitral
p.259 | Notas
